In [1]:
import os
import glob
import numpy as np
import pandas as pd
import pickle
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 13})  # Adjust the font size as desired

# Frame-level Inference

Inferences are made using the deep learning model at every frame of the input video

In [15]:
def plot_confusion_matrix(labels, predictions, title, label_names, filename):
    cm = confusion_matrix(labels, predictions)
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues", xticklabels=label_names, yticklabels=label_names, annot_kws={"size": 22}, cbar=False)
    plt.xlabel('Predicted', fontsize=16, weight='bold')
    plt.ylabel('True', fontsize=16, weight='bold')
    plt.title(title, fontsize=18, weight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=16, weight='bold')
    plt.yticks(rotation=0, fontsize=16, weight='bold')
    plt.tight_layout()  # Adjust layout to fit labels
    os.makedirs(os.path.join(os.getcwd(), "output"), exist_ok=True)
    plt.savefig(os.path.join(os.getcwd(), "output", filename))
    plt.close()

In [16]:
L10_path = "/data1/GraphModellingExperiments/L10/IEEEPaper/C3D/w30-o29/confusion-matrix-unseenTest.pkl"
L10_label_names = ["Labelling-I", "Labelling-II", "Position MB", "Scan MB\nLabels", "Insert PCIe\nFillers", "Insert CSSD\nCard", "Push-Secure\nMotherboard", "Route\nCabling", "Get Next\nChassis", "Miscellaneous"]
AssemblyDemo_path = "/data1/GraphModellingExperiments/AssemblyDemo/IEEEPaper/Detectron2/w30-o29/confusion-matrix-unseenTest.pkl"
AssemblyDemo_label_names = ["Position\nMotherboard", "Attach\nBracket", "Secure\nMotherboard", "Insert\nCard", "Attach\nDevice", "Remove\nBattery", "Miscellaneous"]

# Load the pickle files
L10_data = pickle.load(open(L10_path, "rb"))
AssemblyDemo_data = pickle.load(open(AssemblyDemo_path, "rb"))

plot_confusion_matrix(L10_data["labels"], L10_data["predictions"], title = "Dataset-I", label_names=L10_label_names, filename="cm-d1-c3d.png")
plot_confusion_matrix(AssemblyDemo_data["labels"], AssemblyDemo_data["predictions"], title = "Dataset-II", label_names=AssemblyDemo_label_names, filename="cm-d2-d2.png")

# Impact of testing proportions

- The proportion of data available was varied for testing for every training cycle
- Proportions are [2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
    - The proportions can change 

In [ ]:
# Results directory
assembly_type = "AssemblyDemo"
results_dir = f"/data1/GraphModellingExperiments/{assembly_type}/TrainEval"
classes_type = "WithHands"
results_dir = os.path.join(results_dir, classes_type)
# Temporal information 
temporal_classes = ["1", "2", "6"]
temporal_classes.insert(0, "sc")

# Type of test
test_type = "unseenTest"


In [ ]:
# Final directory
data_dir = os.path.join(results_dir, "-".join(temporal_classes))
# Get all files inside
data_items = glob.glob(data_dir + os.sep + "*")

# Load all the csv files
df_dict = {}
for data_item in data_items:
    
    # Find all the csv files within
    csv_files_path = glob.glob(data_item + os.sep + "**" + os.sep + "*" + test_type + ".csv", recursive=True)
    df_dict[data_item.split(os.sep)[-1]] = {}
    for csv_file in csv_files_path:
        # Get the proportion
        proportion = int(csv_file.split(os.sep)[-2].split("-")[-1])
        df_dict[data_item.split(os.sep)[-1]][proportion] = pd.read_csv(csv_file, header="infer", index_col=0)


In [ ]:
# Plotting the results
metric_to_consider = "macro avg"
# proportions = [2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
proportions = [15, 30, 50]
df_parsed_data = {}
for wo_config in df_dict.keys():
    df_parsed_data[wo_config] = []
    for proportion in proportions:
        df_parsed_data[wo_config].append(df_dict[wo_config][proportion].loc[metric_to_consider]["f1-score"])

# Plots
fig = plt.figure(figsize=(10, 8))
axs = fig.add_subplot([0, 0, 1, 1])

ordered_key_items = ["w30-o0", "w30-o10", "w30-o15", "w30-o25", "w15-o14", "w30-o29"]
for index, wo_config in enumerate(ordered_key_items):
    if wo_config not in df_parsed_data.keys():
        continue
    axs.plot(proportions, df_parsed_data[wo_config], label=wo_config, linewidth=3, color=plt.cm.tab10(index))
axs.legend()
axs.set_xticks(proportions)
axs.set_ylim([0.80, 0.96])
axs.set_xlabel("Proportions of test data", fontsize=14)
axs.set_ylabel("Macro avg. f1-score", fontsize=14)
axs.set_title(f"Macro avg. f1 vs Proportions of test data (Temporal Class { '-'.join(temporal_classes)} )", fontsize=16)

# Impact of Overlap rate

In [ ]:
print(f"The class used for inference is {'-'.join(temporal_classes)}")

In [ ]:
# Get the average for a class
wo_averaged = {}
for wo_config in df_parsed_data.keys():
    wo_averaged[wo_config] = np.mean(df_parsed_data[wo_config][2:])


In [ ]:
# Bar charts for a class
# Plots
fig = plt.figure(figsize=(10, 8))
axs = fig.add_subplot([0, 0, 1, 1])

ordered_key_items = ["w30-o0", "w30-o10", "w30-o15", "w30-o25", "w15-o14", "w30-o29"]
for index, wo_config in enumerate(ordered_key_items):
    if wo_config not in wo_averaged.keys():
        continue
    
    # Compute the proportion
    overlap_rate = round(int(wo_config.split("-")[1][1:])/int(wo_config.split("-")[0][1:]), 2)
    
    axs.bar(wo_config + f"({overlap_rate})", wo_averaged[wo_config], width=0.5, color=plt.cm.tab10(index))

# Axis information
axs.set_xlabel("Overlap rate", fontsize=14)
axs.set_ylabel("F1-Score (Macro-Avg)", fontsize=14)
axs.set_title(f"Macro-Avg F1-Score vs Overlap Rate ({'-'.join(temporal_classes)})", fontsize=16)


# Compare between the two classes

In [ ]:
# Results directory
results_dir = f"/data1/GraphModellingExperiments/{assembly_type}/TrainEval"
classes_type = "WithHands"
results_dir = os.path.join(results_dir, classes_type)
# Temporal information 
temporal_classes_compared = ["1-2", "1-2-6"]
temporal_classes_compared = ["sc-" + x for x in temporal_classes_compared]

# Type of test
test_type = "unseenTest"

In [ ]:
# Final directory
data_dirs = [os.path.join(results_dir, x) for x in temporal_classes_compared]

df_dict_class_combined = {}
for data_dir in data_dirs:
    # Get all files inside
    data_items = glob.glob(data_dir + os.sep + "*")
    
    # Load all the csv files
    df_dict = {}
    for data_item in data_items:
        
        # Find all the csv files within
        csv_files_path = glob.glob(data_item + os.sep + "**" + os.sep + "*" + test_type + ".csv", recursive=True)
        df_dict[data_item.split(os.sep)[-1]] = {}
        for csv_file in csv_files_path:
            # Get the proportion
            proportion = int(csv_file.split(os.sep)[-2].split("-")[-1])
            df_dict[data_item.split(os.sep)[-1]][proportion] = pd.read_csv(csv_file, header="infer", index_col=0)
            
    # Add by classes
    df_dict_class_combined[data_dir.split(os.sep)[-1]] = df_dict

In [ ]:
# Plotting the results
metric_to_consider = "macro avg"
# proportions = [2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
proportions = [15, 30, 50]

df_parsed_class_combined = {}
for class_type in df_dict_class_combined.keys():
    df_dict = df_dict_class_combined[class_type]
    df_parsed_data = {}
    for wo_config in df_dict.keys():
        df_parsed_data[wo_config] = []
        for proportion in proportions:
            df_parsed_data[wo_config].append(df_dict[wo_config][proportion].loc[metric_to_consider]["f1-score"])
            
    df_parsed_class_combined[class_type] = df_parsed_data

In [ ]:
# Get the average of the scores 
averaged_values = {}
for class_type in df_parsed_class_combined.keys():
    averaged_values[class_type] = {}
    for wo_config in df_parsed_class_combined[class_type].keys():
        averaged_values[class_type][wo_config] = np.mean(df_parsed_class_combined[class_type][wo_config][2:])


In [ ]:
# Plotting
# Plots
fig = plt.figure(figsize=(10, 8))
axs = fig.add_subplot([0, 0, 1, 1])

# Set the bar parameters
bar_width = 0.2

# Get the similar items across the two categories
for index, class_type in enumerate(averaged_values.keys()):
    if index == 0:
        items = list(averaged_values[class_type].items())
    else:
        if len(items) > len(list(averaged_values[class_type].items())):
            items = list(averaged_values[class_type].items())

# ordered_key_items = ["w30-o0", "w30-o10", "w30-o15", "w30-o25", "w15-o14", "w30-o29"]
ordered_key_items = ["w30-o15", "w30-o25", "w15-o14", "w30-o29"]
categories = []
classes_type = list(averaged_values.keys())
for index, wo_config in enumerate(ordered_key_items):
    
    availability = all([wo_config in averaged_values[x] for x in averaged_values.keys()])
    if not availability:
        continue
    
    # Compute the proportion
    overlap_rate = round(int(wo_config.split("-")[1][1:])/int(wo_config.split("-")[0][1:]), 2)
    categories.append(wo_config + f"({overlap_rate})")
    
    # Creating bars - For all the class types considered
    for i, class_type in enumerate(classes_type):
        values = averaged_values[class_type][wo_config] 
        axs.bar(index + i * bar_width, values, width=bar_width, color=plt.cm.tab10(i), edgecolor='black', label=class_type, linewidth=3,)
    
# Axis information
axs.set_xlabel("Overlap rate", fontsize=14)
axs.set_ylabel("F1-Score (Macro-Avg)", fontsize=14)
axs.set_title(f"Macro-Avg F1-Score vs Overlap Rate across classes ({'|'.join(classes_type)})", fontsize=16)
plt.xticks([r + bar_width/2 for r in range(len(categories))], categories)
axs.legend(classes_type)
    

# Impact of testing proportions for each temporal class

In [ ]:
# Results directory
results_dir = "/data1/GraphModellingExperiments/L10/TrainEval"
classes_type = "WithoutHands"
results_dir = os.path.join(results_dir, classes_type)
# Temporal information 
temporal_classes = ["11", "12", "11-12"]
temporal_classes = ["sc-" + x for x in temporal_classes]

# Type of test
test_type = "unseenTest"

In [ ]:
# Final directory
data_dirs = [os.path.join(results_dir, x) for x in temporal_classes]

df_dict_class_combined = {}
for data_dir in data_dirs:
    # Get all files inside
    data_items = glob.glob(data_dir + os.sep + "*")
    
    # Load all the csv files
    df_dict = {}
    for data_item in data_items:
        
        # Find all the csv files within
        csv_files_path = glob.glob(data_item + os.sep + "**" + os.sep + "*" + test_type + ".csv", recursive=True)
        df_dict[data_item.split(os.sep)[-1]] = {}
        for csv_file in csv_files_path:
            # Get the proportion
            proportion = int(csv_file.split(os.sep)[-2].split("-")[-1])
            df_dict[data_item.split(os.sep)[-1]][proportion] = pd.read_csv(csv_file, header="infer", index_col=0)
            
    # Add by classes
    df_dict_class_combined[data_dir.split(os.sep)[-1]] = df_dict

In [ ]:
selected_overlap = "w30-o25"
proportions = [2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
selected_values = {}
for class_instance in df_dict_class_combined.keys():
    selected_values[class_instance] = []
    for proportion in proportions:
        selected_values[class_instance].append(df_dict_class_combined[class_instance][selected_overlap][proportion].loc["macro avg"]["f1-score"])
    

In [ ]:
# Getting the indices
indices = np.arange(len(selected_values['sc-11']))

# Plotting the grouped bar chart
plt.figure(figsize=(10, 6))

# Plot each key as a group
for i, (key, values) in enumerate(selected_values.items()):
    plt.bar(indices + 0.2*i, values, width=0.2, label=key)

plt.xlabel('Proportions')
plt.ylabel('Macro Avg F1-Score')
plt.title('Variation across different test proportion')
plt.xticks(indices + 0.2*(len(selected_values)-1)/2, proportions)
plt.legend()
plt.show()

# Comparison between WithHands and WithoutHands

In [ ]:
assembly_type = "AssemblyDemo"
results_dir = f"/data1/GraphModellingExperiments/{assembly_type}/TrainEval"

# List all files inside
all_files = glob.glob(os.path.join(results_dir, "**/*unseenTest.csv"), recursive=True)

In [ ]:
# Go through files and group together
results = {}
for file in all_files:
    # Get the information
    temporal_class = file.split(os.sep)[-4]
    overlap_rate = file.split(os.sep)[-3]
    
    if overlap_rate not in results.keys():
        results[overlap_rate] = {}
        
    if temporal_class not in results[overlap_rate].keys():
        results[overlap_rate][temporal_class] = []
        
    results[overlap_rate][temporal_class].append(pd.read_csv(file, header="infer", index_col=0).loc["macro avg"]["f1-score"])
    
# Average them
data = {outer_key: {inner_key: sum(inner_list) / len(inner_list) for inner_key, inner_list in outer_value.items()} for outer_key, outer_value in results.items()}
    

In [ ]:
outer_keys = list(data.keys())
overlap_rates = {wo_config: round(int(wo_config.split("-")[1][1:]) / int(wo_config.split("-")[0][1:]), 2) for wo_config in outer_keys}
outer_keys = sorted(outer_keys, key=lambda x: overlap_rates[x])

categories = []
for wo_config in outer_keys:
    overlap_rate = round(int(wo_config.split("-")[1][1:])/int(wo_config.split("-")[0][1:]), 2)
    categories.append(wo_config + f"({overlap_rate})")

inner_keys = list(data[outer_keys[0]].keys())
num_inner_keys = len(inner_keys)
bar_width = 0.2
index = range(len(outer_keys))

fig = plt.figure(figsize=(10, 8))
axs = fig.add_subplot([0, 0, 1, 1])

ordered_key_items = ["w30-o15", "w30-o25", "w15-o14", "w30-o29"]
for i in range(num_inner_keys):
    values = [data[key][inner_keys[i]] for key in outer_keys]
    axs.bar([x + i * bar_width for x in index], values, bar_width, label=inner_keys[i], color=plt.cm.tab10(i), edgecolor='black', linewidth=3,)

axs.set_xlabel('Overlap Rate', fontsize=14)
axs.set_ylabel('Macro Avg F1-Score', fontsize=14)
axs.set_title('Macro-Avg F1-Score across overlap rates and temporal classes', fontsize=16)
axs.set_ylim([0, 1.0])
plt.xticks([x + (num_inner_keys - 1) * bar_width / 2 for x in index], categories)
axs.legend()

plt.show()